# Recommendations: phase 1

Proving the algorithm from `docs/specs/recommendations.md` before any migration or UI.
The logic lives in `api/recommender/`; this notebook loads the snapshot, calls into it, and looks at the results.

**Exit criterion:** with its *starting* parameters, the pipeline beats every baseline on end-to-end MRR.

Clear all outputs before committing. The snapshot in `recommender/snapshot/` is gitignored.

## Setup

In [ ]:
# Re-import edited modules before each cell runs, so changes to recommender/*.py
# show up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# The kernel's working directory is this notebook's folder; api/ is its parent.
API_DIR = Path.cwd().parent
sys.path.insert(0, str(API_DIR))
SNAPSHOT = API_DIR / "recommender" / "snapshot"

from recommender.library import RATING_SCORES, is_eligible, load_library  # noqa: E402

## 1. The library

In [ ]:
library = load_library(SNAPSHOT)
games = pd.DataFrame([vars(g) for g in library.played])
# Int64 is pandas' nullable integer; plain int64 has no NA, so ids would turn to floats.
games["igdb_id"] = games["igdb_id"].astype("Int64")
games["score"] = games["rating"].map(RATING_SCORES)

print(f"{len(games)} played, {len(library.wishlist)} wishlisted")
print(f"{len(library.rated)} rated and IGDB-backed; eligible: {is_eligible(library)}")
games.head()

In [ ]:
# How the ratings are spread, best to worst. A lopsided curve is what
# mean-centering (algorithm step 2) is for.
games["rating"].replace("", "(unrated)").value_counts().reindex(
    [*RATING_SCORES, "(unrated)"], fill_value=0
)

In [ ]:
# Hand-entered games have no igdb_id, so no IGDB features: they drop out of
# the taste model and are excluded from candidates by name instead.
games[games["igdb_id"].isna()][["name", "system", "rating"]]

In [ ]:
# Preview of step 2: each rating's weight relative to your own mean.
# Positives are w > 0; negatives need w < 0 AND a rating of Okay or Bad.
# The mean is over the rated, IGDB-backed games only, as the taste model sees them.
mean = sum(g.score for g in library.rated) / len(library.rated)
pd.DataFrame({"score": RATING_SCORES, "w": {r: s - mean for r, s in RATING_SCORES.items()}}).assign(
    role=lambda d: d.apply(
        lambda row: (
            "positive" if row.w > 0 else "negative" if row.name in ("Okay", "Bad") else "ignored"
        ),
        axis=1,
    )
)

## 2. The IGDB snapshot

Built by `uv run python -m recommender.snapshot` (run from `api/`): the library's IGDB rows, their `similar_games`, and the candidate pool.

In [ ]:
import json

igdb = {row["id"]: row for row in json.loads((SNAPSHOT / "igdb_games.json").read_text())}
pool = json.loads((SNAPSHOT / "pool.json").read_text())
print(f"{len(igdb)} games cached, fetched {pool['fetched_at']}")
print(f"popular {len(pool['popular'])}, upcoming {len(pool['upcoming'])}")

In [ ]:
# Feature coverage on the library side: a field most games lack can't carry much weight.
fields = [
    "genres",
    "themes",
    "keywords",
    "player_perspectives",
    "game_modes",
    "involved_companies",
    "franchises",
    "collections",
    "similar_games",
]
counts = pd.DataFrame({f: [len(igdb[g.igdb_id].get(f, [])) for g in library.rated] for f in fields})
pd.DataFrame({"has any": (counts > 0).mean().round(2), "median": counts.median()})

In [ ]:
# The recall ceiling: of the positives, how many would the pool contain if that
# game were held out? A miss here is a miss end-to-end however good the scorer is.
# A port or bundle counts as reached through its main game(s).
from recommender.versions import build_main_game_index, canonical_ids

index = build_main_game_index(igdb)
mean = sum(g.score for g in library.rated) / len(library.rated)
positives = [g for g in library.rated if g.score > mean]
popular = set(pool["popular"])
mains = {g.igdb_id: canonical_ids(g.igdb_id, igdb, index) for g in positives}


def reachable(held):
    neighbours = {
        s
        for g in positives
        if g is not held
        for m in mains[g.igdb_id]
        for s in igdb[m].get("similar_games", [])
    }
    return any(m in popular or m in neighbours for m in mains[held.igdb_id])


missed = [g.name for g in positives if not reachable(g)]
print(f"reachable: {len(positives) - len(missed)}/{len(positives)}")
missed